In [0]:
CREATE OR REPLACE TABLE eatery_group.silver.restaurant_clean AS
SELECT
  restaurant_id,
  TRIM(restaurant_name) AS restaurant_name,
  country_code,

  INITCAP(city) AS city,
  INITCAP(locality) AS locality,
  TRIM(address) AS address,
  locality_verbose,

  TRY_CAST(latitude AS DECIMAL(10,6)) AS latitude,
  TRY_CAST(longitude AS DECIMAL(10,6)) AS longitude,

  LOWER(TRIM(cuisines)) AS cuisines,

  CASE WHEN average_cost_for_two < 0 THEN NULL 
       ELSE average_cost_for_two 
  END AS average_cost_for_two,

  currency,

  CASE WHEN has_table_booking = 'Yes' THEN 1 ELSE 0 END AS has_table_booking,
  CASE WHEN has_online_delivery = 'Yes' THEN 1 ELSE 0 END AS has_online_delivery,
  CASE WHEN is_delivering_now = 'Yes' THEN 1 ELSE 0 END AS is_delivering_now,
  CASE WHEN switch_to_order_menu = 'Yes' THEN 1 ELSE 0 END AS switch_to_order_menu,

  price_range,

  CASE WHEN aggregate_rating BETWEEN 0 AND 5 
       THEN aggregate_rating 
       ELSE NULL 
  END AS aggregate_rating,

  rating_color,
  rating_text,

  COALESCE(votes, 0) AS votes,

  ingestion_timestamp,
  source_file

FROM eatery_group.silver.restaurant_validation
WHERE
  invalid_id = 0
  AND invalid_name = 0
  AND invalid_city = 0
  AND invalid_cuisines = 0
  AND invalid_latitude = 0
  AND invalid_longitude = 0
  AND invalid_cost = 0
  AND invalid_votes = 0
  AND invalid_rating = 0
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY restaurant_id
  ORDER BY ingestion_timestamp DESC
) = 1;

SELECT * FROM eatery_group.silver.restaurant_clean;